# Qwen3-8B on Kaggle T4 x2 — gates, memory, and every optimization re-measured at 8B

Run top-to-bottom in a fresh **GPU T4 x2** session, Internet **On**. Every command is fail-fast.
GPU 0 serves; GPU 1 is reserved for the dequantised reference and (later) the 0.6B draft.

The predictions are in [`docs/qwen3-8b-optimization-map.md`](../docs/qwen3-8b-optimization-map.md)
and are repeated in the results cell beside each measurement. The model is served at **W4A16**
(4-bit grouped weights, fp16 activations) because 16.4 GB of fp16 weights do not fit a 15 GB card;
the identity gate therefore compares the Triton kernels against torch over the *same packed
weights* (`reference_mode`), so a gate failure is a kernel, not quantisation. Quantisation error is
a separate question, not measured here.

Budget: gates ~15 min (plus the 16 GB checkpoint download), then ~12 min per A/B setting.
Trim `AB_PLAN` to fit the session.

In [ ]:
import json, os, shlex, subprocess
from datetime import datetime, timezone
from pathlib import Path
RUN = Path('results/sm75/qwen3-8b-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
RUN.mkdir(parents=True, exist_ok=False)
os.environ['RUN'] = str(RUN.resolve())
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
def run(command, gpu=0):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    print('+', f'[gpu {gpu}]', command, flush=True)
    subprocess.run(command, shell=True, check=True, env=env)
print(RUN)

In [ ]:
run('bash scripts/setup_kaggle.sh')
run('python scripts/kaggle_preflight.py --out "$RUN/preflight.json"')
run('git rev-parse HEAD > "$RUN/git_sha.txt" && git status --short > "$RUN/git_status.txt"')
run('nvidia-smi --query-gpu=index,name,memory.total,clocks.max.sm --format=csv | tee "$RUN/gpu.txt"')

In [ ]:
MODEL = "Qwen/Qwen3-8B"
MAX_MODEL_LEN = 4096        # served max context: warm-up buckets AND admission
MAX_ACTIVE = 8
BUCKETS = "1 2 4 8"
COMMON = (f"--model {MODEL} --quantize w4a16 --num-blocks auto --max-model-len {MAX_MODEL_LEN} "
          f"--max-active {MAX_ACTIVE} --prefill-graph-rows 4 --fused-graph-prefill-rows 2")
# The order is the prediction order: the two expected reversals first.
AB_PLAN = [
    ("fused_step",     "chat"), ("fused_step",     "long"),
    ("prefill_chunk",  "long"),
    ("kv_dtype",       "long"),
    ("cuda_graphs",    "chat"),
    ("prefix_cache",   "chat"),
    ("triton_rmsnorm", "chat"), ("triton_rope", "chat"), ("triton_swiglu", "chat"),
    ("decode_split_k", "long"),
]
REPEATS, DURATION = 3, 30

In [ ]:
# Gate 1: static checks and the CPU suite (includes the W4A16 install/loader tests).
run('python -m pytest -q -m "not cuda" -p no:cacheprovider')
# Gate 2: CUDA suite on GPU 0 - kernels against their fp16 shadows, on the 0.6B fixture.
run('python -m pytest -q -m cuda -p no:cacheprovider', gpu=0)

In [ ]:
# Gates 3+4 on the 8B itself: loads W4A16 without an fp16 detour, sizes the pool from free
# memory, captures the lean graph set, and runs the live token gate. `problems` must be [].
run(f'python scripts/check_hooks.py {COMMON} --buckets {BUCKETS} --out "$RUN/check_hooks_8b.json"', gpu=0)
hooks = json.loads((RUN / 'check_hooks_8b.json').read_text())
print('problems:', hooks.get('problems'))
assert not hooks.get('problems'), 'live token gate failed; stop here'

In [ ]:
for setting, profile in AB_PLAN:
    out = f'$RUN/ab_{setting}_{profile}.json'
    run(f'python -m benchmarks.reliability.ab --setting {setting} --prompt-profile {profile} '
        f'{COMMON} --cuda-graphs --concurrency {MAX_ACTIVE} --repeats {REPEATS} --duration {DURATION} '
        f'--out "{out}"', gpu=0)

In [ ]:
import glob
import pandas as pd

PREDICTED = {
    ("fused_step", "chat"):     "large win (saves a 15 GB weight read per prefill-carrying step)",
    ("fused_step", "long"):     "large win",
    ("prefill_chunk", "long"):  "TTFT/ITL trade 14x sharper than at 0.6B",
    ("kv_dtype", "long"):       "INT8 KV resolves for the first time (KV ~16% of step bytes)",
    ("cuda_graphs", "chat"):    "~1.3x, down from 3.5x",
    ("prefix_cache", "chat"):   "grows (13.9 GFLOP per reused token)",
    ("triton_rmsnorm", "chat"): "shrinks ~3x in share",
    ("triton_rope", "chat"):    "unresolved",
    ("triton_swiglu", "chat"):  "unresolved",
    ("decode_split_k", "long"): "batch-1/long-context only",
}
KEY = ["step_timing.expected_gap_ms", "latency.itl_p50", "latency.itl_p99", "latency.ttft_p50",
       "step_timing.decode_step_p50_ms", "step_timing.prefill_step_p50_ms"]
rows = []
for path in sorted(glob.glob(f"{RUN}/ab_*.json")):
    r = json.loads(open(path).read())
    setting, profile = r["setting"], r["prompt_profile"]
    row = {"setting": setting, "profile": profile,
           "predicted": PREDICTED.get((setting, profile), ""),
           "arms": " vs ".join(r["arm_settings"]), "skipped": list(r.get("skipped_arms", {})),
           "gate": "ok" if not r["token_identity"]["early_divergence_arms"] else "FAILED STOCK GATE"}
    for m in KEY:
        row[m.split(".")[-1]] = r["comparison"].get(m)
    rows.append(row)
table = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 60)
print("=== Qwen3-8B W4A16 on T4: prediction vs measurement (variant vs baseline arm) ===")
display(table)
table.to_csv(RUN / "predictions_vs_results.csv", index=False)

print("\n=== memory and graph set from the live gate ===")
for k in ("quantization", "quantization_report", "memory", "warmup", "graph_replay_report"):
    if k in hooks: print(k, ":", hooks[k])

## Reading it

- A `fused_step` win here with a loss at 0.6B is the architecture of the prediction working: the
  saving scaled with weight bytes, the overhead did not.
- `FAILED STOCK GATE` on any row means a kernel disagrees with torch over the same packed weights
  within 8 tokens. That is a bug, never quantisation — quantisation cannot differ from itself.
- `unresolved` is a result. Write it beside the prediction either way.
- Not in this notebook: perplexity of W4A16 against fp16 (needs the fp16 model in layers on
  GPU 1), speculative decoding with the 0.6B draft, and the vLLM AWQ comparison at matched KV.

In [ ]:
import shutil; print(shutil.make_archive('/kaggle/working/qwen3_8b_round', 'zip', RUN))